In [4]:
from pathlib import Path

import uproot
import awkward as ak
import numpy as np

# Data filename. If the file moves, update this value or add a candidate path below.
DATA_FILENAME = "JetNtuple_RunIISummer16_13TeV_MC_1.root"
candidate_paths = [
    Path.cwd() / DATA_FILENAME,
    Path.cwd() / "QCDJetsMachineLearning" / DATA_FILENAME,
    Path.cwd().parent / "QCDJetsMachineLearning" / DATA_FILENAME,
]
ROOT_PATH = next((path.resolve() for path in candidate_paths if path.is_file()), None)
if ROOT_PATH is None:
    searched = "\n".join(f"  - {path.resolve()}" for path in candidate_paths)
    raise FileNotFoundError(f"Could not find {DATA_FILENAME}. Searched:\n{searched}")

# Open the ROOT file and locate its TTree/RNTuple objects.
f = uproot.open(ROOT_PATH)

# Inspect all directories, not only the top level.
classnames = f.classnames(recursive=True)

tree_candidates = [
    k for k, cls in classnames.items()
    if "TTree" in cls or "RNTuple" in cls
]

if not tree_candidates:
    raise ValueError(f"No TTree or RNTuple was found in {ROOT_PATH.name}")

def cycle_num(key):
    return int(key.rsplit(";", 1)[1]) if ";" in key else 0

tree_key = max(tree_candidates, key=cycle_num)
tree = f[tree_key]

print(f"Data file : {ROOT_PATH}")
print(f"Tree      : {tree_key}")
print(f"Entries   : {tree.num_entries:,}")
print(f"Branches  : {len(tree.keys())}")


Data file : D:\OneDrive - The University of Manchester\summer2026\SCIML\jet praticales\QCDJetsMachineLearning\JetNtuple_RunIISummer16_13TeV_MC_1.root
Tree      : AK4jets/jetTree;3
Entries   : 28,614
Branches  : 66


In [8]:
# PyTorch Geometric is required: pip install torch torch-geometric
import torch
from torch_geometric.data import Data

K_NEIGHBORS = 8
particle_feature_names = ("pT", "dEta", "dPhi")

# Read jet labels, event IDs, particle features, and the constituent mask.
branches = [
    "event",
    "physFlav",
    "isPhysUDS",
    "isPhysG",
    "PF_pT",
    "PF_dEta",
    "PF_dPhi",
    "PF_fromAK4Jet",
]
raw_data = tree.arrays(branches, library="ak")

# Keep only jets with one unambiguous binary label.
is_quark = raw_data["isPhysUDS"] == 1
is_gluon = raw_data["isPhysG"] == 1
selected_jets = raw_data[is_quark ^ is_gluon]


def build_knn_edges(pos, k=K_NEIGHBORS):
    """Return directed k-nearest-neighbour edges using only native PyTorch."""
    num_nodes = pos.size(0)
    if num_nodes < 2:
        return torch.empty((2, 0), dtype=torch.long)

    k = min(k, num_nodes - 1)
    distances = torch.cdist(pos, pos)
    distances.fill_diagonal_(float("inf"))
    neighbours = distances.topk(k, largest=False).indices

    target = torch.arange(num_nodes).repeat_interleave(k)
    source = neighbours.reshape(-1)
    return torch.stack((source, target), dim=0)


# Create one PyTorch Geometric Data object per jet.
jet_graphs = []
for jet in selected_jets:
    constituent_mask = ak.to_numpy(jet["PF_fromAK4Jet"]) == 1
    particle_matrix = np.column_stack(
        [
            ak.to_numpy(jet["PF_pT"])[constituent_mask],
            ak.to_numpy(jet["PF_dEta"])[constituent_mask],
            ak.to_numpy(jet["PF_dPhi"])[constituent_mask],
        ]
    )

    x = torch.as_tensor(particle_matrix, dtype=torch.float32)
    pos = x[:, 1:3].clone()
    edge_index = build_knn_edges(pos)

    graph = Data(
        x=x,
        pos=pos,
        edge_index=edge_index,
        y=torch.tensor([int(jet["isPhysG"])], dtype=torch.long),
        original_flavor=torch.tensor([int(jet["physFlav"])], dtype=torch.long),
        event_id=torch.tensor([int(jet["event"])], dtype=torch.long),
    )
    jet_graphs.append(graph)

print(f"PyG graphs created: {len(jet_graphs):,} / {len(raw_data):,} jets")
print("Node feature columns:", particle_feature_names)
print("Target convention: 0 = light quark (UDS), 1 = gluon")
print(jet_graphs[0])


d:\OneDrive - The University of Manchester\summer2026\SCIML\jet praticales\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyG graphs created: 13,912 / 28,614 jets
Node feature columns: ('pT', 'dEta', 'dPhi')
Target convention: 0 = light quark (UDS), 1 = gluon
Data(x=[26, 3], edge_index=[2, 208], y=[1], pos=[26, 2], original_flavor=[1], event_id=[1])


# PyTorch Geometric Jet Graphs

The second code cell now converts every selected jet into one `torch_geometric.data.Data` object. The complete filtered collection is stored in `jet_graphs`.

## Graph fields

Each jet graph contains:

- `x`: node-feature matrix with shape `[num_particles, 3]`; columns are `[pT, dEta, dPhi]`.
- `pos`: particle coordinates with shape `[num_particles, 2]`; columns are `[dEta, dPhi]`.
- `edge_index`: directed k-nearest-neighbour graph with shape `[2, num_edges]`.
- `y`: graph-level target, where `0` is a light-quark (UDS) jet and `1` is a gluon jet.
- `original_flavor`: the original `physFlav` value, retained only as metadata.
- `event_id`: the source event identifier, retained for leakage-safe dataset splitting.

Only particles satisfying `PF_fromAK4Jet == 1` become graph nodes. Each particle connects to its nearest `K_NEIGHBORS` particles in the `(dEta, dPhi)` plane. The edge builder uses native PyTorch, so `torch-cluster` is not required.

## Dataset status

`jet_graphs` still contains the complete selected sample; it has not yet been divided into training, validation, and test sets. The later split should group by `event_id` so jets from the same event never appear in different subsets. After splitting, the graph lists can be passed to `torch_geometric.loader.DataLoader`.


In [ ]:
features = [
    # jet-level
    "jetPt",
    "jetEta",
    "jetPhi",
    "jetMass",
    "jetGirth",
    "jetArea",
    "nPF",

    # particle-level
    "PF_pT",
    "PF_dR",
    "PF_dPhi",
    "PF_dEta",
    "PF_mass",
]

arrays = tree.arrays(
    features,
    entry_stop=5000,
    library="ak"
)

print("\nLoaded", len(arrays), "jets")

jet_index = 0

print("\nJet 0:")
for feature in features:
    print(feature, "=", arrays[feature][jet_index])